# ✈️ Remaining Useful Life (RUL) of Turbofan Engines with NASA C-MAPSS (FD001)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/02_predictive_maintenance/06_remaining_useful_life/n18_cmapss_rul_prediction.ipynb)

> **En Colab:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (opcional: los modelos son pequeños y en CPU tarda unos minutos).

Up to now we have **detected** failures or anomalies. Predictive maintenance also wants to **predict the future**:
*how many operating cycles are left before this engine fails?* This is the **Remaining Useful Life (RUL)**, a **regression** problem on **multivariate time series** (Parte 2, slide 12).

**Dataset:** NASA **C-MAPSS** (Saxena et al., 2008), sub-dataset **FD001**: 100 training engines run until failure and 100 test engines whose series stop some time before failure.
Each cycle has 3 operating settings and 21 sensors (temperatures, pressures, speeds...).

We compare three PyTorch models on the same sliding windows:
1. 🛠️ **Hand-crafted features + MLP**: statistics of each window (mean, std, min, max, last value, slope) → the "feature engineering" approach (Parte 2, slide 139).
2. 🌊 **1D-CNN**: convolutions along time (Parte 2, slides 160–170).
3. 🔁 **LSTM**: recurrent network with memory (Parte 2, slides 171–174).

Metrics: **RMSE** and the asymmetric **NASA scoring function**, which penalises **late** predictions (predicting more life than there really is) much more than early ones.

In [ ]:
# ⚙️ Setup cell (run first). Detects Google Colab and installs only what may be missing there; locally it does nothing.
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
# 📦 Import Required Libraries
import os
import io
import copy
import time
import zipfile
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'   # only for automatic tests
RUL_CAP = 125        # piece-wise linear target: RUL is clipped at 125 cycles (a healthy engine does not "degrade" yet)
WINDOW = 30          # length of each input window (cycles)
EPOCHS = 3 if QUICK_RUN else 40
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
VAL_ENGINES = 20     # training engines kept apart for validation (split BY ENGINE, never by window)
print('QUICK_RUN =', QUICK_RUN, '| EPOCHS =', EPOCHS)

## 📥 Download C-MAPSS into ./data
Sources (no login needed): Kaggle mirror via `kagglehub` → public NASA/PHM Society copy on S3. Manual option: download `CMAPSSData.zip` from the
[NASA Prognostics Data Repository](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/) and unzip it inside `./data`.

In [ ]:
DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))
CMAPSS_S3 = 'https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip'
SUBSET = 'FD001'


def find_cmapss(root):
    hits = list(Path(root).rglob(f'train_{SUBSET}.txt'))
    return hits[0].parent if hits else None


def download_cmapss():
    if find_cmapss(DATA_DIR):
        return find_cmapss(DATA_DIR)
    errors = []
    try:
        import kagglehub
        folder = find_cmapss(kagglehub.dataset_download('behrad3d/nasa-cmaps'))
        if folder:
            return folder
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    try:
        with urllib.request.urlopen(CMAPSS_S3, timeout=120) as r:
            outer = zipfile.ZipFile(io.BytesIO(r.read()))
        inner_name = next(n for n in outer.namelist() if n.endswith('CMAPSSData.zip'))
        zipfile.ZipFile(io.BytesIO(outer.read(inner_name))).extractall(DATA_DIR / 'CMAPSSData')
        if find_cmapss(DATA_DIR):
            return find_cmapss(DATA_DIR)
    except Exception as e:  # noqa: BLE001
        errors.append(f'S3: {e}')
    raise RuntimeError('Could not download C-MAPSS:\n' + '\n'.join(errors))


CMAPSS_DIR = download_cmapss()
COLS = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]
read = lambda name: pd.read_csv(CMAPSS_DIR / name, sep=r'\s+', header=None, names=COLS)  # noqa: E731
train_df, test_df = read(f'train_{SUBSET}.txt'), read(f'test_{SUBSET}.txt')
rul_test_last = pd.read_csv(CMAPSS_DIR / f'RUL_{SUBSET}.txt', sep=r'\s+', header=None).iloc[:, 0].values
print(f'Data folder: {CMAPSS_DIR}')
print(f'train: {train_df.shape} ({train_df.unit.nunique()} engines) | test: {test_df.shape} ({test_df.unit.nunique()} engines)')
train_df.head()

## 🎯 Build the RUL target

- **Training and validation engines** run until failure → at cycle `t` of an engine whose last cycle is `T`: `RUL = T − t`. For these targets we clip at `RUL_CAP = 125` because early in life the sensors may not reveal the exact time to failure.
- **Test engines** stop before failure; `RUL_FD001.txt` gives the original, uncapped RUL at the last cycle. Test RMSE and NASA score are computed against this real value, not the clipped target.


In [ ]:
def add_rul(df, rul_last=None):
    df = df.copy()
    last = df.groupby('unit')['cycle'].transform('max')
    extra = 0 if rul_last is None else df['unit'].map(dict(zip(range(1, len(rul_last) + 1), rul_last)))
    df['RUL_raw'] = last - df['cycle'] + extra
    df['RUL'] = df['RUL_raw'].clip(upper=RUL_CAP)
    return df

train_df = add_rul(train_df)
test_df = add_rul(test_df, rul_test_last)
test_df['RUL'] = test_df['RUL_raw']  # keep the true, uncapped test target for final evaluation
lengths = train_df.groupby('unit').cycle.max()
print(f'Training engine life: min={lengths.min()} mean={lengths.mean():.0f} max={lengths.max()} cycles')
print(f'Test: true RUL at last cycle min={rul_test_last.min()} max={rul_test_last.max()} | '
      f'shortest test series = {test_df.groupby("unit").cycle.max().min()} cycles')

unit = 1
fig, ax = plt.subplots(figsize=(7, 3))
d = train_df[train_df.unit == unit]
ax.plot(d.cycle, d.RUL_raw, 'k:', label='linear RUL (T - t)')
ax.plot(d.cycle, d.RUL, 'r', label=f'clipped training target (cap = {RUL_CAP})')
ax.set_xlabel('cycle'); ax.set_ylabel('RUL'); ax.set_title(f'Target for training engine {unit}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 🔎 Which sensors carry information?
In FD001 there is a single operating condition, so several sensors are (almost) **constant** and are useless. We drop every column whose standard deviation in training is ~0.

In [ ]:
# Split training engines into train and validation before fitting preprocessing steps.
rng = np.random.default_rng(SEED)
units = train_df.unit.unique()
val_units = rng.choice(units, VAL_ENGINES, replace=False)
tr_df = train_df[~train_df.unit.isin(val_units)].copy()
va_df = train_df[train_df.unit.isin(val_units)].copy()

# Keep non-constant sensors using training engines only; apply this same feature set to val/test.
sensor_candidates = [f's{i}' for i in range(1, 22)]
stds = tr_df[sensor_candidates].std()
FEATURES = [s for s in sensor_candidates if stds[s] > 1e-3]
print('Dropped constant sensors (training engines only):', [s for s in sensor_candidates if s not in FEATURES])
print('Used sensors:', FEATURES)

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, s in zip(axes.ravel(), FEATURES[:8]):
    for u in tr_df.unit.unique()[:5]:
        d = tr_df[tr_df.unit == u]
        ax.plot(d.RUL_raw, d[s], lw=0.8)
    ax.invert_xaxis(); ax.set_title(s); ax.set_xlabel('cycles to failure')
plt.suptitle('Sensor evolution for training engines (x axis: cycles before failure)')
plt.tight_layout()
plt.show()


## 🧪 Split by engine, select sensors, normalise and build sliding windows

- **Split by engine first**: windows from the same engine are similar and must not be shared between train and validation.
- The sensor selection and **StandardScaler** are fitted only on the training engines; the resulting features and scaler are then applied unchanged to validation and test.
- Each sample is a window of `WINDOW` consecutive cycles × `len(FEATURES)` sensors; its target is the clipped RUL at the last cycle for train/validation. Test targets remain original and uncapped.
- Test uses one window per engine (the last `WINDOW` cycles), as required by the benchmark. Series shorter than the window are padded with their first value.


In [ ]:
scaler = StandardScaler().fit(tr_df[FEATURES])
for d in (tr_df, va_df, test_df):
    d[FEATURES] = scaler.transform(d[FEATURES])


def engine_windows(d, last_only=False):
    """Windows ending at every cycle (or only at the last cycle) of one engine."""
    x, y = d[FEATURES].values.astype(np.float32), d['RUL'].values.astype(np.float32)
    if len(x) < WINDOW:
        x = np.vstack([np.repeat(x[:1], WINDOW - len(x), axis=0), x])
        y = np.concatenate([np.repeat(y[:1], WINDOW - len(y)), y])
    ends = [len(x)] if last_only else range(WINDOW, len(x) + 1)
    return np.stack([x[e - WINDOW:e] for e in ends]), np.array([y[e - 1] for e in ends])


def make_windows(df, last_only=False):
    xs, ys = zip(*(engine_windows(g, last_only) for _, g in df.groupby('unit')))
    return np.concatenate(xs), np.concatenate(ys)

Xw_tr, yw_tr = make_windows(tr_df)
Xw_va, yw_va = make_windows(va_df)
Xw_te, yw_te = make_windows(test_df, last_only=True)
print(f'train windows: {Xw_tr.shape} | val windows: {Xw_va.shape} | test (last window per engine): {Xw_te.shape}')


## 🛠️ Model 1 input: hand-crafted features per window
For every sensor in the window: **mean, std, min, max, last value and slope** (linear trend). 6 statistics × 15 sensors = 90 features → a small MLP.

In [ ]:
def handcrafted(Xw):
    t = np.arange(WINDOW, dtype=np.float32)
    t = (t - t.mean()) / ((t - t.mean()) ** 2).sum()                  # least-squares slope weights
    slope = np.einsum('t,ntf->nf', t, Xw)
    return np.concatenate([Xw.mean(1), Xw.std(1), Xw.min(1), Xw.max(1), Xw[:, -1, :], slope], axis=1).astype(np.float32)


F_tr, F_va, F_te = handcrafted(Xw_tr), handcrafted(Xw_va), handcrafted(Xw_te)
f_scaler = StandardScaler().fit(F_tr)
F_tr, F_va, F_te = (f_scaler.transform(a).astype(np.float32) for a in (F_tr, F_va, F_te))
print('Hand-crafted feature matrix:', F_tr.shape)

## 🧠 The three models

In [ ]:
N_SENS = len(FEATURES)


class FeatureMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):                 # x: (B, n_features)
        return self.net(x).squeeze(1)


class CNN1D(nn.Module):
    def __init__(self, n_sensors, window):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_sensors, 32, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(64 * window, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):                 # x: (B, T, sensors) -> Conv1d wants (B, channels=sensors, T)
        return self.head(self.conv(x.transpose(1, 2))).squeeze(1)


class LSTMRegressor(nn.Module):
    def __init__(self, n_sensors, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):                 # x: (B, T, sensors)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(1)   # hidden state at the last time step


MODELS = {
    'Hand-crafted + MLP': (lambda: FeatureMLP(F_tr.shape[1]), (F_tr, F_va, F_te)),
    '1D-CNN': (lambda: CNN1D(N_SENS, WINDOW), (Xw_tr, Xw_va, Xw_te)),
    'LSTM': (lambda: LSTMRegressor(N_SENS), (Xw_tr, Xw_va, Xw_te)),
}
for name, (make, _) in MODELS.items():
    print(f'{name:<20} parameters: {sum(p.numel() for p in make().parameters()):,}')

## 📏 Metrics: RMSE and NASA score

With the error $d = \widehat{RUL} - RUL$ (positive = **late**: we think there is more life than there really is):

$$ s = \sum_i \begin{cases} e^{-d_i/13} - 1 & d_i < 0 \ \text{(early)} \\ e^{d_i/10} - 1 & d_i \ge 0 \ \text{(late)} \end{cases} $$

A late prediction can mean an **in-flight failure**; an early one only means maintaining a bit too soon. Lower is better for both metrics.

In [ ]:
def rmse(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def nasa_score(pred, true):
    d = pred - true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))


d = np.linspace(-40, 40, 200)
plt.figure(figsize=(6, 3))
plt.plot(d, np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1))
plt.axvline(0, color='k', lw=0.5)
plt.xlabel('error d = predicted RUL - true RUL'); plt.ylabel('penalty'); plt.title('NASA scoring function (one engine)')
plt.text(-38, 15, 'early\n(maintain too soon)'); plt.text(12, 15, 'late\n(risk of failure!)', color='red')
plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

## 🏋️ Train the three models
MSE loss on the target divided by `RUL_CAP` (values around 0–1 help optimisation). We keep the epoch with the lowest **validation RMSE**.

In [ ]:
@torch.no_grad()
def predict(model, X):
    model.eval()
    out = [model(xb.to(device)).cpu() for (xb,) in DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=1024)]
    return torch.cat(out).numpy() * RUL_CAP


def train_regressor(make, Xtr, Xva, name):
    torch.manual_seed(SEED)
    model = make().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(yw_tr / RUL_CAP)), batch_size=BATCH_SIZE, shuffle=True)
    hist, best, best_state, start = {'train_rmse': [], 'val_rmse': []}, np.inf, None, time.time()
    for _epoch in range(EPOCHS):
        model.train()
        tot = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tot += loss.item() * len(yb)
        val_rmse = rmse(predict(model, Xva), yw_va)
        hist['train_rmse'].append(np.sqrt(tot / len(yw_tr)) * RUL_CAP); hist['val_rmse'].append(val_rmse)
        if val_rmse < best:
            best, best_state = val_rmse, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    hist['time'] = time.time() - start
    print(f'{name:<20} {EPOCHS} epochs in {hist["time"]:6.1f}s | best val RMSE = {best:.2f}')
    return model, hist

trained, histories, rows = {}, {}, []
for name, (make, (Xtr, Xva, _Xte)) in MODELS.items():
    model, hist = train_regressor(make, Xtr, Xva, name)
    trained[name], histories[name] = model, hist
    val_pred = np.clip(predict(model, Xva), 0, None)
    rows.append({'model': name, 'parameters': sum(q.numel() for q in model.parameters()), 'train time (s)': hist['time'],
                 'val RMSE': rmse(val_pred, yw_va), 'val NASA score': nasa_score(val_pred, yw_va)})
results = pd.DataFrame(rows)
print()
print('Validation metrics only (targets clipped at RUL_CAP):')
print(results.round(2).to_string(index=False))


In [ ]:
# 📈 Learning curves
plt.figure(figsize=(7, 4))
for name, h in histories.items():
    line, = plt.plot(h['val_rmse'], label=f'{name} - validation')
    plt.plot(h['train_rmse'], ls='--', color=line.get_color(), alpha=0.6, label=f'{name} - train')
plt.xlabel('Epoch'); plt.ylabel('RMSE (cycles)'); plt.ylim(0, 60); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('Learning curves')
plt.tight_layout()
plt.show()

## ⏳ RUL trajectories on validation engines
We slide the window over validation engines to inspect predictions across their life cycle. Test performance is held back until the final evaluation, after model and safety-margin selection using validation only.


In [ ]:
def trajectory(df_engine, name):
    Xw, yw = engine_windows(df_engine)
    X_in = f_scaler.transform(handcrafted(Xw)).astype(np.float32) if name == 'Hand-crafted + MLP' else Xw
    return df_engine['cycle'].values[-len(yw):], yw, np.clip(predict(trained[name], X_in), 0, None)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, u in zip(axes.ravel(), val_units[:6]):
    d = va_df[va_df.unit == u]
    for k, name in enumerate(MODELS):
        cyc, true, pred = trajectory(d, name)
        if k == 0:
            ax.plot(cyc, true, 'k-', lw=2, label='true clipped RUL')
        ax.plot(cyc, pred, lw=1, label=name)
    ax.set_title(f'validation engine {u}'); ax.set_xlabel('cycle'); ax.set_ylabel('RUL'); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 🛡️ Select the model and safety margin using validation only
Because late predictions are costly, a safety margin can be subtracted from the predicted RUL. We select the model using validation NASA score (validation RMSE breaks ties), then choose the margin by validation NASA score. The test set is not used for either decision.


In [ ]:
best_row = results.sort_values(['val NASA score', 'val RMSE'], ascending=True).iloc[0]
best_name = best_row['model']
margins = np.arange(0, 31, 2)
Xva_in = MODELS[best_name][1][1]
p_va = np.clip(predict(trained[best_name], Xva_in), 0, None)
sc_val = [nasa_score(np.clip(p_va - m, 0, None), yw_va) for m in margins]
rm_val = [rmse(np.clip(p_va - m, 0, None), yw_va) for m in margins]
best_margin = int(margins[int(np.argmin(sc_val))])

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(margins, sc_val, 'r-o', label='NASA score (validation)')
ax2 = ax1.twinx()
ax2.plot(margins, rm_val, 'b-s', label='RMSE (validation)')
ax1.set_xlabel('safety margin subtracted (cycles)'); ax1.set_ylabel('NASA score', color='r'); ax2.set_ylabel('RMSE', color='b')
ax1.set_title(f'{best_name}: safety margin selected on validation'); ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Model selected on validation: {best_name} | validation margin: {best_margin} cycles')


### Why not choose a model with test metrics?

The test set is reserved for an unbiased final estimate of generalisation. Choosing a model or safety margin after looking at test results leaks information from the test set and makes that estimate optimistic. Use validation to make decisions, then evaluate once on test.


In [ ]:
# Final test evaluation: run only after model and margin selection are fixed.
test_preds, final_rows = {}, []
for _, row in results.iterrows():
    name = row['model']
    Xte_in = MODELS[name][1][2]
    pred = np.clip(predict(trained[name], Xte_in), 0, None)
    test_preds[name] = pred
    final_rows.append({
        'model': name,
        'val RMSE': row['val RMSE'],
        'val NASA score': row['val NASA score'],
        'test RMSE (raw RUL)': rmse(pred, yw_te),
        'NASA score (test, raw RUL)': nasa_score(pred, yw_te),
        'late predictions (%)': 100 * np.mean(pred > yw_te),
        'train time (s)': row['train time (s)'],
    })
final_results = pd.DataFrame(final_rows)
print('Final comparison: validation targets are clipped; test targets are original and uncapped.')
print(final_results.round(2).to_string(index=False))

selected_pred = test_preds[best_name]
margin_pred = np.clip(selected_pred - best_margin, 0, None)
print(f'\nSelected model: {best_name} | validation-selected margin: {best_margin} cycles')
print(f'Selected model, test without margin: RMSE={rmse(selected_pred, yw_te):.2f} | NASA score={nasa_score(selected_pred, yw_te):.1f}')
print(f'Selected model, test with margin:    RMSE={rmse(margin_pred, yw_te):.2f} | NASA score={nasa_score(margin_pred, yw_te):.1f}')


## 📝 Summary

- Train and validation targets are clipped at `RUL_CAP`; the final test metrics use the original, uncapped RUL.
- Split engines before selecting sensors or fitting preprocessing. Fit sensor selection and normalization on training engines only, then apply them unchanged to validation and test.
- Choose the model and safety margin using validation RMSE and NASA score only. Test is held out for a single final evaluation and must not influence model selection.
- Hand-crafted features need domain knowledge but are simple and fast; the 1D-CNN learns local temporal patterns; the LSTM keeps a memory of the sequence.
- **RMSE** treats early and late errors the same; the **NASA score** does not. A model with a slightly worse RMSE can be preferable if it is late less often.

🔧 Things you can try (and report in the quiz): change `WINDOW` (15, 30, 50), `RUL_CAP` (100, 125, 150), the number of `EPOCHS`, or train with an **asymmetric loss** that penalises late errors more.
